# LC-conditioned 1D Laplacian eigenvalue PINN

**Problem.** $-u''(x) = \lambda u(x)$ on $\Omega = (0, \pi)$ with $u(0) = u(\pi) = 0$.

**Analytic reference.** $u_k(x) = \sqrt{2/\pi} \sin(kx)$, $\lambda_k = k^2$, $k = 1, 2, \ldots$

**Motivation.** Notebook `06_laplace_1d` trains $K$ independent small networks, one per eigenmode. Natural LC-PINN extension: condition a *single* network on the mode index $k$ so that $u_\theta(x, k)$ returns the $k$-th eigenfunction. Here $k$ plays the role that $\lambda$ plays in the loss-weighted LC-PINN — the network input that picks out which element of the solution family we want. If it works, the paper narrative generalises from "condition on loss weights" to "condition on any problem parameter".

**Method (Ky Fan variational principle).** One shared network with hard Dirichlet BC,
$$
u_\theta(x, k_{\rm enc}) = x(\pi - x) \cdot N_\theta(x, k_{\rm enc}),
$$
loss
$$
\mathcal{L} = \sum_{k=1}^{K} w_k \cdot R\!\left(u_\theta(\cdot, k)\right) + \alpha \sum_{i<j} \cos^2\!\left(\angle\left(u_\theta(\cdot, i), u_\theta(\cdot, j)\right)\right),
$$
where $R(u) = \mathbb{E}[|u'|^2]/\mathbb{E}[u^2]$ is the Rayleigh quotient and $w_k = 1/k$. The decreasing weights break the permutation symmetry of $\sum R$ so slot $k$ maps to the $k$-th eigenfunction in order (Ky Fan characterisation of $\sum_k \lambda_k$). The $\cos^2$ orthogonality term is scale-invariant, the same fix as in `06_laplace_1d` — Rayleigh is scale-invariant, so $(\langle u_i, u_j\rangle)^2$ is escaped by shrinking $\|u\|$.

**Three design choices that all mattered.**
1. **One-hot $k$ encoding** (DIM_LAMBDA = K). Earlier attempts with $k_{\rm norm} \in [0, 1]$ collapsed adjacent slots — eigenvalues right but eigenfunctions mixed. One-hot gives each slot its own affine projection of the first layer.
2. **$\cos^2$ orthogonality** (not $\langle u_i, u_j\rangle^2$) — reason above.
3. **Curriculum**: split training into $K$ equal phases, phase $j$ uses only slots $1..j$ in the loss. Mirrors sequential deflation with shared weights: mode 1 locks in during phase 1, then mode 2 is added while mode 1 is maintained by its Rayleigh term + pairwise $\cos^2$, and so on.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from pinns.device import select_device, device_info
from pinns.equations import laplace_1d as lap       # sequential baseline
from pinns.equations import laplace_1d_lc as lap_lc

device = select_device()
print(f"Device: {device_info(device)}")

REPO = pathlib.Path.cwd().parent
torch.manual_seed(0); np.random.seed(0)

## 1. Train the LC-conditioned network

Config below matches `scripts/run_laplace_1d_lc.py --K 5 --n-epochs 100000 --curriculum --alpha-orth 100 --seed 0`. Expect ~20 min on MPS.

Skip this cell and jump to **§2** if you have already run the script — a checkpoint is loaded from `checkpoints/laplace_1d_lc.pt`.

In [ ]:
K = 5
N_EPOCHS = 100_000
LR = 1e-3
ALPHA_ORTH = 100.0
W_EXP = 1.0
N_INTERIOR = 1024
HIDDEN_DIMS = [64, 64, 64, 64]

TRAIN_NOW = False  # set True to retrain from scratch

model = lap_lc.LCEigenmodeNet(K_max=K, hidden_dims=HIDDEN_DIMS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"LCEigenmodeNet params: {n_params:,}  (encoding={model.encoding})")

history = None
if TRAIN_NOW:
    t0 = time.time()
    history = lap_lc.train_lc_eigenmode(
        model, K=K, device=device,
        n_epochs=N_EPOCHS, lr=LR,
        alpha_orth=ALPHA_ORTH, n_interior=N_INTERIOR,
        w_exp=W_EXP, curriculum=True, log_every=500,
    )
    print(f"Trained in {(time.time()-t0)/60:.1f} min")
else:
    ckpt = torch.load(REPO / "checkpoints" / "laplace_1d_lc.pt", map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded checkpoint (K={ckpt['K']}, hidden={ckpt['hidden_dims']})")

## 2. Eigenvalues and eigenfunctions vs analytic

In [ ]:
results_lc = lap_lc.evaluate_all(model, K, device, nx=500)
order = lap_lc.reorder_by_rayleigh(model, K, device, nx=500)

print(f"Ordering check (slot -> rank-by-Rayleigh): {order}")
print(f"{'k':>3} {'lambda_true':>12} {'lambda_hat':>12} {'|Dl|/l':>8} {'rel-L2 u':>10}")
print('-' * 50)
for r in results_lc:
    print(f"{r['k']:>3} {r['lambda_true']:>12.4f} {r['lambda_hat']:>12.4f} "
          f"{r['lambda_rel_err']:>8.4f} {r['rel_l2']:>10.4f}")

## 3. Figure: eigenfunctions, LC vs analytic

In [ ]:
nx = 500
x_eval = np.linspace(lap.X_MIN, lap.X_MAX, nx)

fig, axes = plt.subplots(1, K, figsize=(3.2 * K, 3), sharey=True)
for k_idx in range(K):
    k = k_idx + 1
    u_ref, _ = lap.reference_eigenmode(k, x_eval)
    u_pred = lap_lc.predict_eigenmode(model, k, K, x_eval, device)
    u_pred_n = lap_lc._l2_normalise(u_pred)
    if float(np.dot(u_pred_n, u_ref)) < 0.0:
        u_pred_n = -u_pred_n
    ax = axes[k_idx]
    r = results_lc[k_idx]
    ax.plot(x_eval, u_ref, 'k-', lw=1.5, label='analytic')
    ax.plot(x_eval, u_pred_n, 'r--', lw=1.3, label='LC-PINN')
    ax.set_title(f"k={k},  rel-L2={r['rel_l2']:.4f}\n"
                 f"lambda_hat={r['lambda_hat']:.3f} (true {r['lambda_true']:.0f})")
    ax.set_xlabel('x'); ax.grid(alpha=0.3)
    if k_idx == 0:
        ax.legend(fontsize=8)
fig.suptitle(f"LC-conditioned eigenfunctions (single network, K={K})", y=1.03)
plt.tight_layout()
plt.show()

## 4. Side-by-side comparison with the sequential baseline

Sequential results come from `results/laplace_1d_results.json` (the Apr 21 run of `scripts/run_laplace_1d.py`).

In [ ]:
seq_path = REPO / 'results' / 'laplace_1d_results.json'
if seq_path.exists():
    seq = json.loads(seq_path.read_text())
    results_seq = seq['results']
    print(f"{'k':>3} | {'lambda_true':>10} | "
          f"{'lambda_seq':>10} {'rel-L2 seq':>10} | "
          f"{'lambda_LC':>10} {'rel-L2 LC':>10}")
    print('-' * 72)
    for r_lc, r_seq in zip(results_lc, results_seq):
        print(f"{r_lc['k']:>3} | {r_lc['lambda_true']:>10.4f} | "
              f"{r_seq['lambda_hat']:>10.4f} {r_seq['rel_l2']:>10.4f} | "
              f"{r_lc['lambda_hat']:>10.4f} {r_lc['rel_l2']:>10.4f}")
else:
    print(f'Sequential JSON not found at {seq_path}. Run scripts/run_laplace_1d.py first.')

## 5. Takeaways

- **Eigenvalues:** LC recovers all 5 within ~5%, beating sequential on 4/5 modes. The Ky Fan-penalty + curriculum combination gives accurate spectra from a single shared-weight network.
- **Eigenfunctions:** LC is competitive for $k = 1..3$ (rel-L2 ~2–7%), degrades to ~28% for $k = 4, 5$. Shape contamination between higher-$k$ slots is the dominant error source — even with one-hot $k$, the shared MLP trunk leaks some mode-$k$ energy into mode-$(k{\pm}1)$.
- **Paper narrative.** LC-PINN as a generic "condition-on-any-parameter" framework: loss weights → mode index → (future) PDE coefficients. Eigenvalue recovery at single-network cost is a clean story; high-$k$ eigenfunction accuracy is a legitimate next-step problem (richer encodings / hypernetworks), not a dealbreaker.
- **Cost.** One network, 13k params, 20 min on MPS, queried at any $k \in \{1..5\}$ in a single forward pass.